In [1]:
using Pkg
using Metal

cd(@__DIR__)
Pkg.activate("../")
ParamFile = "../config/testparam.csv"  # maybe GeoPoints and planet1D should be fusioned

# batchGPU should be at this level (I have not made it as a module yet, since the choice of Metal/CUDA should be done in a manual way)
include("../src/batchFiles/batchGPU.jl")


include("../src/commonBatchs.jl")
include("../src/planet1D.jl")
include("../src/GeoPoints.jl")

include("../src/flexOPT.jl")

using .commonBatchs, .planet1D, .GeoPoints, .flexOPT

  Activating project at `~/Documents/Github/flexOPT`


devs = Metal.devices() = Metal.MTL.MTLDeviceInstance[Metal.MTL.MTLDeviceInstance (object of type AGXG13XDevice)]
→ Using Metal backend (1 device(s))
Selected backend type: MetalBackend


input parameters

In [ ]:
#famousEquationType="2DacousticTime"
#Δ = (1.0,1.0,1.0)
#famousEquationType="1DsismoFreqHomo" # GT95
famousEquationType="1DsismoTimeHomo" #GT98
#famousEquationType="3DsismoTimeIso" 
Δ = (1.0,1.0)

# orders: -1 -> indicator function, 0 -> box car, >=1 -> B-spline

orderBtime=1
orderBspace=1

pointsInSpace=3
pointsInTime=3

# the order of errors to be controlled
supplementaryOrder=2

# new parameters for interpolated Taylor expansion μ for field

fieldItpl = (ptsSpace = 1,ptsTime = 1,offsetSpace=1,offsetTime=1,YorderBspace=-1,YorderBtime=1) #offsetSpace and offsetTime ∈ z 
# μ points should be distributed from y_min+offset*Δy to y_max-offset*Δy offset can be negative too


# new parameters for interpolated Taylor expansion μᶜ for material
materItpl = (ptsSpace = 1,ptsTime = 1,offsetSpace=1,offsetTime=1,YorderBspace=-1,YorderBtime=1)



(ptsSpace = 1, ptsTime = 1, offsetSpace = 1, offsetTime = 1, YorderBspace = -1, YorderBtime = 1)

In [3]:
concreteParametersForOPTConstruction = @strdict famousEquationType Δ orderBtime orderBspace pointsInSpace pointsInTime supplementaryOrder fieldItpl materItpl


Dict{String, Any} with 9 entries:
  "fieldItpl"          => (ptsSpace = 1, ptsTime = 1, offsetSpace = 1, offsetTi…
  "Δ"                  => (1.0, 1.0, 1.0, 1.0)
  "supplementaryOrder" => 2
  "materItpl"          => (ptsSpace = 1, ptsTime = 1, offsetSpace = 1, offsetTi…
  "orderBspace"        => 1
  "orderBtime"         => 1
  "famousEquationType" => "1DsismoTimeHomo"
  "pointsInSpace"      => 3
  "pointsInTime"       => 3

In [4]:
optRec = makeOPTsemiSymbolic(concreteParametersForOPTConstruction)
recette = optRec["recette"]

(vars, iVars) = ((ρ, μ), 1)
(vars, iVars) = ((ρ, μ), 2)
(vars, iVars) = ((ρ, μ), 1)
(vars, iVars) = ((ρ, μ), 2)
pointsIndices = availablePointsConfigurations[iConfigGeometry] = SVector{2, Int64}[[1, 1] [1, 2] [1, 3]; [2, 1] [2, 2] [2, 3]; [3, 1] [3, 2] [3, 3]]
middleLinearν = centrePointConfigurations[iConfigGeometry] = 5
μPoints = availableμPoints[iConfigGeometry] = SVector{2, Float64}[[2.0, 2.0];;]
μᶜPoints = availableμᶜPoints[iConfigGeometry] = SVector{2, Float64}[[2.0, 2.0];;]
μaxes = availableμaxes[iConfigGeometry] = ([2.0], [2.0])
μᶜaxes = availableμᶜaxes[iConfigGeometry] = ([2.0], [2.0])
size(μPoints) = (1, 1)
pointν = pointsIndices[middleLinearν] = [2, 2]


┌ Info: File /Users/nobuaki/Documents/Github/flexOPT/data/WYYKKIntegralSymbolic/WYYKKIntegralSymbolic_5eca38d4.jld2 does not exist. Producing it now...
└ @ DrWatson /Users/nobuaki/.julia/packages/DrWatson/2QF5p/src/saving_files.jl:106
┌ Warning: The Git repository ('/Users/nobuaki/Documents/Github/flexOPT') is dirty! Appending -dirty to the commit ID.
└ @ DrWatson /Users/nobuaki/.julia/packages/DrWatson/2QF5p/src/saving_tools.jl:71
┌ Info: File /Users/nobuaki/Documents/Github/flexOPT/data/WYYKKIntegralSymbolic/WYYKKIntegralSymbolic_5eca38d4.jld2 saved.
└ @ DrWatson /Users/nobuaki/.julia/packages/DrWatson/2QF5p/src/saving_files.jl:115
┌ Info: File /Users/nobuaki/Documents/Github/flexOPT/data/WYYKKIntegralSymbolic/WYYKKIntegralSymbolic_dccfb816.jld2 does not exist. Producing it now...
└ @ DrWatson /Users/nobuaki/.julia/packages/DrWatson/2QF5p/src/saving_files.jl:106
┌ Warning: The Git repository ('/Users/nobuaki/Documents/Github/flexOPT') is dirty! Appending -dirty to the commit ID.
└ @ 

(typeof(μPoints), μPoints[1], typeof(pointsIndices)) = (Matrix{SVector{2, Float64}}, [2.0, 2.0], Matrix{SVector{2, Int64}})


┌ Info: File /Users/nobuaki/Documents/Github/flexOPT/data/taylorCoefInv/taylorCoefInv_61b76839.jld2 does not exist. Producing it now...
└ @ DrWatson /Users/nobuaki/.julia/packages/DrWatson/2QF5p/src/saving_files.jl:106


typeof(pointsIndices) = Matrix{SVector{2, Int64}}
(multiOrdersIndices, pointsIndices, μpointsIndices) = (CartesianIndices((5, 5)), SVector{2, Int64}[[1, 1] [1, 2] [1, 3]; [2, 1] [2, 2] [2, 3]; [3, 1] [3, 2] [3, 3]], SVector{2, Float64}[[2.0, 2.0];;])


DimensionMismatch: DimensionMismatch: arrays could not be broadcast to a common size

In [5]:
Ajiννᶜ = optRec["recette"].lhs.Ajiννᶜ

UndefVarError: UndefVarError: `optRec` not defined in `Main`
Suggestion: add an appropriate import or assignment. This global was declared but not assigned.